In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from PIL import Image
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
# Handling images (stored in 3 folders in MyDrive)
image_root_path = "/content/drive/MyDrive/ML_project/images/"
image_path_mapping = {}
for root, dirs, files in os.walk(image_root_path):
  for file in files:
      if file.endswith(".png"):
          image_path_mapping[file] = os.path.join(root, file)

print(f"Indexing complete: {len(image_path_mapping)} PNG images found.")

Indexing complete: 2298 PNG images found.


In [4]:
# 1. Configuration & Hyperparameters
CONFIG = {
    "image_size": 224,
    "batch_size": 32,
    "lr": 0.0001,
    "epochs": 20,
    "n_splits": 5,  # For K-Fold Cross Validation
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

In [5]:
# 2. Data Preprocessing (Metadata)
def preprocess_metadata(df):
    df = df.copy()

    # Fill missing values for clinical data
    df['age'] = df['age'].fillna(df['age'].mean())
    df['smoke'] = df['smoke'].fillna(df['smoke'].mode()[0])
    df['drink'] = df['drink'].fillna(df['drink'].mode()[0])

    # Identify ALL non-numeric columns to encode them
    categorical_cols = df.select_dtypes(include=['object']).columns

    le = LabelEncoder()
    for col in categorical_cols:
        if col not in ['patient_id', 'lesion_id', 'img_id', 'diagnostic']:
            df[col] = le.fit_transform(df[col].astype(str))

    # Scale numerical data (Age) - Removed to prevent data leakage
    #scaler = StandardScaler()
    #df[['age']] = scaler.fit_transform(df[['age']])

    # Final safety check: fill any remaining NaNs with 0
    df = df.fillna(0)

    return df

In [6]:
# 3. Custom Multimodal Dataset
class SkinCancerDataset(Dataset):
    def __init__(self, df, path_mapping, transform=None):
        self.df = df
        self.path_mapping = path_mapping
        self.transform = transform
        self.label_mapping = {label: i for i, label in enumerate(df['diagnostic'].unique())}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['img_id']

        if not img_id.endswith(".png"):
            img_id += ".png"

        img_path = self.path_mapping.get(img_id)

        if img_path is None:
            raise FileNotFoundError(f"Image {img_id} non trouvée dans les sous-dossiers !")

        image = Image.open(img_path).convert('RGB')

        metadata = self.df.iloc[idx].drop(['patient_id', 'lesion_id', 'img_id', 'diagnostic']).values.astype(np.float32)
        label = self.label_mapping[self.df.iloc[idx]['diagnostic']]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(metadata), torch.tensor(label)

In [7]:
# 4. Multimodal Fusion Model (CNN + MLP)
class MultimodalNet(nn.Module):
    def __init__(self, num_classes, num_metadata_features):
        super(MultimodalNet, self).__init__()

        # Image Branch (ResNet50)
        self.cnn = resnet50(weights=ResNet50_Weights.DEFAULT)
        num_ftrs = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity() # Remove final FC layer

        # Metadata Branch (MLP)
        self.mlp = nn.Sequential(
            nn.Linear(num_metadata_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Fusion Layer
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 32, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, img, meta):
        img_features = self.cnn(img) # Shape: [batch, 2048]
        meta_features = self.mlp(meta) # Shape: [batch, 32]

        combined = torch.cat((img_features, meta_features), dim=1)
        output = self.classifier(combined)
        return output


In [8]:
# 5. Training and Evaluation Pipeline
def train_model(model, train_loader, val_loader, criterion, optimizer):
    model.to(CONFIG['device'])

    for epoch in range(CONFIG['epochs']):
        model.train()
        running_loss = 0.0

        for images, metadata, labels in train_loader:
            images, metadata, labels = images.to(CONFIG['device']), metadata.to(CONFIG['device']), labels.to(CONFIG['device'])

            optimizer.zero_grad()
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{CONFIG['epochs']}, Loss: {running_loss/len(train_loader):.4f}")



In [9]:
#6. Feature importance
from sklearn.inspection import permutation_importance

def analyze_feature_importance(model, val_loader, feature_names):
    model.eval()
    all_metas = []
    all_labels = []

    # Collecte des données de validation
    with torch.no_grad():
        for _, metas, labels in val_loader:
            all_metas.append(metas)
            all_labels.append(labels)

    X_meta = torch.cat(all_metas).cpu().numpy()
    y_true = torch.cat(all_labels).cpu().numpy()

    # Fonction de score personnalisée pour PyTorch
    def model_predict(X_np):
        X_tensor = torch.tensor(X_np).to(CONFIG['device'])
        # On simule des images vides (zéros) car on veut tester l'importance des métadonnées seules
        dummy_imgs = torch.zeros((X_np.shape[0], 3, 224, 224)).to(CONFIG['device'])
        with torch.no_grad():
            outputs = model(dummy_imgs, X_tensor)
            return outputs.cpu().numpy().argmax(axis=1)

    # Calcul de l'importance par permutation
    importances = []
    # Extraction simplifiée via les poids du premier layer du MLP
    weights = model.mlp[0].weight.data.abs().mean(dim=0).cpu().numpy()

    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': weights
    }).sort_values(by='Importance', ascending=False)

    print("\n--- Feature Importance (Clinical Metadata) ---")
    print(feature_importance_df)
    return feature_importance_df

In [ ]:
# 7. Main Execution Script
if __name__ == "__main__":

    # Loading the metadata
    path = "/content/drive/MyDrive/ML_project/metadata.csv"

    try:
        df = pd.read_csv(path)
        df = df.drop_duplicates(keep='first')
        print("Dataframe loaded. Shape:", df.shape)
    except:
        print("Check if metadata.csv is in the folder")

    df = preprocess_metadata(df)


    # Image Transformations
    transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # K-Fold Cross Validation
    skf = StratifiedKFold(n_splits=CONFIG['n_splits'])

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['diagnostic'])):
        print(f"--- Training Fold {fold+1} ---")

        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()

        # --- Correcting data leakage (Age normalization scaling bounds)
        scaler = StandardScaler()
        train_df[['age']] = scaler.fit_transform(train_df[['age']])
        val_df[['age']] = scaler.transform(val_df[['age']])
        # ----------------------------------

        train_ds = SkinCancerDataset(train_df, image_path_mapping, transform=transform)
        val_ds = SkinCancerDataset(val_df, image_path_mapping, transform=transform)

        # -- handling class imbalance
        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)
        # ------------------------------------------------------------------------

        num_meta = train_df.drop(['patient_id', 'lesion_id', 'img_id', 'diagnostic'], axis=1).shape[1]
        model = MultimodalNet(num_classes=len(df['diagnostic'].unique()), num_metadata_features=num_meta)

        optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])

        # -- handling class imbalance
        # Calculate frequencies strictly using train_df to protect validation partitions
        train_class_counts = train_df['diagnostic'].value_counts().sort_index().values
        weights = 1. / torch.tensor(train_class_counts, dtype=torch.float)
        weights = weights.to(CONFIG['device'])

        criterion = nn.CrossEntropyLoss(weight=weights)
        # -------------------------------------------------------

        train_model(model, train_loader, val_loader, criterion, optimizer)

        # Feature Importance
        meta_features_names = df.drop(['patient_id', 'lesion_id', 'img_id', 'diagnostic'], axis=1).columns.tolist()
        analyze_feature_importance(model, val_loader, meta_features_names)

        # --- Evaluation Phase ---
        model.eval()
        y_true = []
        y_pred = []

        with torch.no_grad():
            for imgs, metas, labels in val_loader:
                imgs, metas = imgs.to(CONFIG['device']), metas.to(CONFIG['device'])
                outputs = model(imgs, metas)
                preds = torch.argmax(outputs, dim=1)

                y_true.extend(labels.numpy())
                y_pred.extend(preds.cpu().numpy())

        # Generation of the Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)

        # Calculation of Specificity and Sensitivity for each class
        print(f"\nResults for Fold {fold+1}:")

        for i, class_name in enumerate(df['diagnostic'].unique()):
            # True Positives, False Positives, False Negatives, True Negatives
            tp = cm[i, i]
            fp = cm[:, i].sum() - tp
            fn = cm[i, :].sum() - tp
            tn = cm.sum() - (tp + fp + fn)

            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

            print(f"Class [{class_name}]:")
            print(f"  - Sensitivity (Recall): {sensitivity:.4f}")
            print(f"  - Specificity: {specificity:.4f}")

        # Classification Report for Precision and F1-Score
        print("\nFull Classification Report:")
        print(classification_report(y_true, y_pred, target_names=df['diagnostic'].unique()))

Dataframe loaded. Shape: (2298, 26)
--- Training Fold 1 ---


/tmp/ipykernel_6451/335419468.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['smoke'] = df['smoke'].fillna(df['smoke'].mode()[0])
/tmp/ipykernel_6451/335419468.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['drink'] = df['drink'].fillna(df['drink'].mode()[0])


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


Epoch 1/20, Loss: 0.9066
Epoch 2/20, Loss: 0.5912
Epoch 3/20, Loss: 0.4284
Epoch 4/20, Loss: 0.3120
Epoch 5/20, Loss: 0.2229
Epoch 6/20, Loss: 0.2069
Epoch 7/20, Loss: 0.1707
Epoch 8/20, Loss: 0.1133
Epoch 9/20, Loss: 0.1043
Epoch 10/20, Loss: 0.0662
Epoch 11/20, Loss: 0.0674
Epoch 12/20, Loss: 0.0610
Epoch 13/20, Loss: 0.0423
Epoch 14/20, Loss: 0.0368
Epoch 15/20, Loss: 0.0459
Epoch 16/20, Loss: 0.0322
Epoch 17/20, Loss: 0.0276
Epoch 18/20, Loss: 0.0376
Epoch 19/20, Loss: 0.0471
Epoch 20/20, Loss: 0.0271

--- Feature Importance (Clinical Metadata) ---
                Feature  Importance
3     background_mother    0.118999
2     background_father    0.117414
8        cancer_history    0.113291
10    has_sewage_system    0.112788
6                gender    0.112769
5             pesticide    0.111973
20            elevation    0.111676
7   skin_cancer_history    0.109865
13           diameter_1    0.108462
17                 hurt    0.107486
21              biopsed    0.106168
18       